# ComfyUI Notebook: FLUX.2-klein-9B (GGUF)

This notebook installs ComfyUI and runs **FLUX.2-klein-9B GGUF** with a quantized **Qwen3-8B GGUF** text encoder and official flux2 VAE.

Recommended flow:
1) Run Cell 1 to install everything and log in to Hugging Face.
2) In Cell 2, keep defaults or adjust quants.
3) Run Cell 3 to download models and test workflow JSON.
4) Run Cell 4 for size/VRAM verification.
5) Run Cell 5 and open the public ComfyUI URL.


In [ ]:
# @title 1) Install ComfyUI + Manager + ComfyUI-GGUF (with swap)
import os

# Small protection against RAM-related crashes
if not os.path.exists('/swapfile'):
    print('Creating swap (8GB)...')
    !sudo fallocate -l 8G /swapfile
    !sudo chmod 600 /swapfile
    !sudo mkswap /swapfile
    !sudo swapon /swapfile
    print('Swap enabled.')

# Useful packages
!apt-get -y update -qq
!apt-get -y install -qq aria2 ffmpeg

# More predictable CUDA allocator (sometimes helps reduce fragmentation)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

# ComfyUI
if not os.path.exists('/content/ComfyUI'):
    print('Cloning ComfyUI...')
    !git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI

%cd /content/ComfyUI

print('Installing python requirements...')
!pip install -U pip
!pip install -r requirements.txt

# Attention acceleration pack (T4-safe)
ENABLE_ACCEL_PACK = True  # Set False to skip optional attention accelerators.

if ENABLE_ACCEL_PACK:
    import sys
    import subprocess

    def pip_try(spec):
        print(f"Installing optional accelerator: {spec}")
        return subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', spec], check=False).returncode == 0

    import torch
    gpu_name = 'CPU'
    cc_major, cc_minor = (0, 0)
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        cc_major, cc_minor = torch.cuda.get_device_capability(0)

    print(f"GPU detected: {gpu_name} (sm{cc_major}{cc_minor})")

    x_ok = pip_try('xformers>=0.0.28.post3')
    print('xformers:', 'OK' if x_ok else 'FAILED (continuing)')

    # FlashAttention-3 is Hopper-only; FlashAttention-2 and SageAttention target Ampere+.
    if cc_major >= 8:
        fa_ok = pip_try('flash-attn>=2.7.0.post2')
        sage_ok = pip_try('sageattention>=2.1.1')
        print('flash-attn:', 'OK' if fa_ok else 'FAILED (continuing)')
        print('sageattention:', 'OK' if sage_ok else 'FAILED (continuing)')
    else:
        print('Skipping flash-attn and sageattention on this GPU: T4/Turing (sm75) uses xformers path.')
else:
    print('Attention acceleration pack disabled by user (ENABLE_ACCEL_PACK=False).')


# ComfyUI-Manager
if not os.path.exists('custom_nodes/ComfyUI-Manager'):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git custom_nodes/ComfyUI-Manager

# ComfyUI-GGUF
if not os.path.exists('custom_nodes/ComfyUI-GGUF'):
    print('Installing ComfyUI-GGUF...')
    !git clone https://github.com/city96/ComfyUI-GGUF.git custom_nodes/ComfyUI-GGUF
    !pip install -r custom_nodes/ComfyUI-GGUF/requirements.txt

# ComfyUI-KJNodes
if not os.path.exists('custom_nodes/comfyui-kjnodes'):
    print('Installing ComfyUI-KJNodes...')
    !git clone https://github.com/kijai/ComfyUI-KJNodes.git custom_nodes/comfyui-kjnodes
if os.path.exists('custom_nodes/comfyui-kjnodes/requirements.txt'):
    !pip install -r custom_nodes/comfyui-kjnodes/requirements.txt



# Hugging Face authentication (interactive prompt)
!pip install -q huggingface_hub
from getpass import getpass
from huggingface_hub import login

hf_token = getpass('Enter Hugging Face token: ').strip()
if not hf_token:
    raise RuntimeError('Hugging Face token is required for this notebook run.')

login(token=hf_token, add_to_git_credential=False)
os.environ['HUGGINGFACE_TOKEN'] = hf_token
os.environ['HF_TOKEN'] = hf_token
print('HF auth: OK')

# Civitai authentication (for LoRA downloads)
civitai_token = getpass('Enter Civitai API token (optional, press Enter to skip): ').strip()
if civitai_token:
    os.environ['CIVITAI_API_TOKEN'] = civitai_token
    print('Civitai auth: OK')
else:
    print('Civitai token not set. Civitai downloads may fail (403).')

print('Done.')


In [ ]:
# @title 2) Settings: GGUF quant selection + size/VRAM table
import math

# ---- Model toggles ----
DOWNLOAD_BASE_MODEL = False
DOWNLOAD_DISTILLED_MODEL = False

# ---- Quant selection (editable) ----
BASE_QUANT = 'Q4_K_M'       # options: Q2_K, Q3_K_S, Q3_K_M, Q4_0, Q4_1, Q4_K_S, Q4_K_M, Q5_0, Q5_1, Q5_K_S, Q5_K_M, Q6_K, Q8_0, BF16, F16
DISTILLED_QUANT = 'Q4_K_M'  # options: Q2_K, Q3_K_S, Q3_K_M, Q4_0, Q4_1, Q4_K_S, Q4_K_M, Q5_0, Q5_1, Q5_K_S, Q5_K_M, Q6_K, Q8_0, BF16, F16

# Qwen3-8B GGUF (text encoder)
QWEN8_QUANT = 'Q4_K_M'  # options: Q2_K, Q2_K_L, Q3_K_S, Q3_K_M, Q4_1, Q4_K_S, Q4_K_M, Q5_K_S, Q5_K_M, Q6_K, Q8_0, IQ4_XS, IQ4_NL, UD-IQ1_S, UD-IQ1_M, UD-IQ2_XXS, UD-IQ2_M, UD-IQ3_XXS, UD-Q2_K_XL, UD-Q3_K_XL, UD-Q4_K_XL, UD-Q5_K_XL, UD-Q6_K_XL, UD-Q8_K_XL, BF16

# If VRAM is limited, keep this enabled to offload text encoder to CPU/RAM
OFFLOAD_TEXT_ENCODER = True

# ---- Auto quant by VRAM ----
AUTO_QUANT_BY_VRAM = True
AUTO_QUANT_VRAM_FRACTION = 0.7
AUTO_VRAM_FALLBACK_GB = 14.0

# ---- File-size tables (GB) from Hugging Face listings ----
MODEL_SIZES_GB = {
  'Q2_K': 3.98,
  'Q3_K_S': 4.69,
  'Q3_K_M': 4.77,
  'Q4_0': 5.62,
  'Q4_1': 6.16,
  'Q4_K_S': 5.83,
  'Q4_K_M': 5.91,
  'Q5_0': 6.71,
  'Q5_1': 7.25,
  'Q5_K_S': 6.94,
  'Q5_K_M': 7.02,
  'Q6_K': 7.87,
  'Q8_0': 9.98,
  'BF16': 18.16,
  'F16': 18.16,
}

QWEN8_SIZES_GB = {
  'Q2_K': 3.28,
  'Q2_K_L': 3.43,
  'Q3_K_S': 3.77,
  'Q3_K_M': 4.12,
  'Q4_1': 5.25,
  'Q4_K_S': 4.80,
  'Q4_K_M': 5.03,
  'Q5_K_S': 5.72,
  'Q5_K_M': 5.85,
  'Q6_K': 6.73,
  'Q8_0': 8.71,
  'IQ4_XS': 4.58,
  'IQ4_NL': 4.79,
  'UD-IQ1_S': 2.28,
  'UD-IQ1_M': 2.40,
  'UD-IQ2_XXS': 2.61,
  'UD-IQ2_M': 3.11,
  'UD-IQ3_XXS': 3.41,
  'UD-Q2_K_XL': 3.50,
  'UD-Q3_K_XL': 4.31,
  'UD-Q4_K_XL': 5.14,
  'UD-Q5_K_XL': 5.88,
  'UD-Q6_K_XL': 7.49,
  'UD-Q8_K_XL': 10.82,
  'BF16': 16.39,
}

FLUX2_VAE_GB = 0.336


def detect_total_vram_gb(default=AUTO_VRAM_FALLBACK_GB):
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.get_device_properties(0).total_memory / (1024**3)
    except Exception as e:
        print(f"VRAM detection failed ({e}); using fallback {default:.2f} GB")
    return default


def _is_supported_quant_name(qname):
    q = str(qname).upper().strip()
    if q.startswith('UD-') or q.startswith('IQ'):
        return False
    if q in {'BF16', 'F16', 'F32'}:
        return True
    return q.startswith('Q')

def _supported_quant_items(size_dict):
    return sorted([(k, v) for k, v in size_dict.items() if _is_supported_quant_name(k)], key=lambda kv: kv[1])

def pick_best_quant(size_dict, budget_gb, overhead=1.10):
    items = _supported_quant_items(size_dict)
    if not items:
        items = sorted(size_dict.items(), key=lambda kv: kv[1])
    fitting = [(k, v) for k, v in items if (v * overhead) <= budget_gb]
    if fitting:
        return fitting[-1][0]
    return items[0][0]


def est_vram_gb(size_gb, overhead=1.10):
    return size_gb * overhead


def print_table(title, d):
    print('\n' + title)
    print('-' * len(title))
    for k, v in d.items():
        print(f"{k:12s}  file~{v:5.2f} GB   VRAM~{est_vram_gb(v):5.2f} GB")


# ---- Auto apply quant choice based on detected VRAM ----
TOTAL_VRAM_GB = detect_total_vram_gb()
AUTO_BUDGET_GB = TOTAL_VRAM_GB * AUTO_QUANT_VRAM_FRACTION
print(f"\nGPU VRAM detected: ~{TOTAL_VRAM_GB:.2f} GB | Auto budget (75%): ~{AUTO_BUDGET_GB:.2f} GB")

if AUTO_QUANT_BY_VRAM:
    if OFFLOAD_TEXT_ENCODER:
        budget_main = max(AUTO_BUDGET_GB - est_vram_gb(FLUX2_VAE_GB), 0.01)
        best = pick_best_quant(MODEL_SIZES_GB, budget_main)
        BASE_QUANT = best
        DISTILLED_QUANT = best
    else:
        best_fit = None
        best_any = None
        for m_k, m_v in _supported_quant_items(MODEL_SIZES_GB):
            for q_k, q_v in _supported_quant_items(QWEN8_SIZES_GB):
                total = est_vram_gb(m_v) + est_vram_gb(q_v) + est_vram_gb(FLUX2_VAE_GB)
                score = (m_v, q_v)
                if (best_any is None) or (total < best_any[0]):
                    best_any = (total, m_k, q_k)
                if total <= AUTO_BUDGET_GB:
                    if (best_fit is None) or (score > best_fit[0]) or (score == best_fit[0] and total < best_fit[1]):
                        best_fit = (score, total, m_k, q_k)

        chosen = best_fit if best_fit is not None else best_any
        BASE_QUANT = chosen[2]
        DISTILLED_QUANT = chosen[2]
        QWEN8_QUANT = chosen[3]

    print(f"Auto quant active: BASE_QUANT={BASE_QUANT}, DISTILLED_QUANT={DISTILLED_QUANT}, QWEN8_QUANT={QWEN8_QUANT}")


print_table('FLUX.2-klein 9B GGUF quants (base + distilled)', MODEL_SIZES_GB)
print_table('Qwen3-8B GGUF (text encoder) quants', QWEN8_SIZES_GB)

print('\nSelected:')
print('  DOWNLOAD_BASE_MODEL      =', DOWNLOAD_BASE_MODEL)
print('  DOWNLOAD_DISTILLED_MODEL =', DOWNLOAD_DISTILLED_MODEL)
print('  BASE_QUANT               =', BASE_QUANT)
print('  DISTILLED_QUANT          =', DISTILLED_QUANT)
print('  QWEN8_QUANT              =', QWEN8_QUANT)
print('  OFFLOAD_TEXT_ENCODER     =', OFFLOAD_TEXT_ENCODER)

shared = est_vram_gb(FLUX2_VAE_GB)
if not OFFLOAD_TEXT_ENCODER:
    shared += est_vram_gb(QWEN8_SIZES_GB[QWEN8_QUANT])

if DOWNLOAD_BASE_MODEL:
    print(f"  Rough VRAM if BASE loaded      : ~{shared + est_vram_gb(MODEL_SIZES_GB[BASE_QUANT]):.2f} GB")
if DOWNLOAD_DISTILLED_MODEL:
    print(f"  Rough VRAM if DISTILLED loaded : ~{shared + est_vram_gb(MODEL_SIZES_GB[DISTILLED_QUANT]):.2f} GB")

print('Note: base and distilled are usually used one-at-a-time in a workflow, not simultaneously.')


# ---- RAM-aware auto guard (Colab-safe) ----
AUTO_QUANT_BY_RAM = True
AUTO_QUANT_RAM_FRACTION = 0.6
AUTO_RAM_OS_RESERVE_GB = 2.0
AUTO_RAM_RUNTIME_GB = 1.2
AUTO_RAM_SPILL_FACTOR = 0.45
AUTO_RAM_GPU_RESIDENT_FACTOR = 0.10
AUTO_RAM_TEXT_OFFLOAD_FACTOR = 1.05
RAM_GUARD_ASSUME_SINGLE_MODEL_RUN = True


def detect_total_ram_gb(default=12.9):
    try:
        import psutil
        return psutil.virtual_memory().total / (1024**3)
    except Exception as e:
        print(f"RAM detection failed ({e}); using fallback {default:.2f} GB")
    return default


def _norm_tokens(name):
    return [t for t in str(name).upper().replace('-', '_').split('_') if t]


def _is_text_quant_var(qvar):
    toks = _norm_tokens(qvar)
    text_keys = {'T5', 'QWEN', 'UMT5', 'FLAN', 'CLIP', 'TEXT', 'ENCODER', 'TE'}
    return any((t in text_keys) or t.startswith('QWEN') or t.startswith('T5') for t in toks)


def _size_dict_for_quant_var(qvar, g):
    pref = qvar[:-6] if qvar.endswith('_QUANT') else qvar
    candidates = [
        f'{pref}_SIZES_GB',
        f'{pref}_SIZE_GB',
        f'{pref}_GGUF_GB',
    ]
    for c in candidates:
        if c in g and isinstance(g[c], dict):
            return c
    # fallback: first dict with matching prefix
    for k, v in g.items():
        if isinstance(v, dict) and k.endswith(('_SIZES_GB', '_GGUF_GB')) and pref in k:
            return k
    return None


def _smallest_quant_key(size_dict):
    if not size_dict:
        return None
    items = _supported_quant_items(size_dict)
    if items:
        return items[0][0]
    return sorted(size_dict.items(), key=lambda kv: kv[1])[0][0]

MIN_QUANT_FLOORS = {
    'T5': 'Q5_K_M',
    'QWEN': 'Q4_K_S',
    'SRPO': 'Q5_K',
    'KLEIN': 'Q5_K_M',
    'ZIMAGE': 'Q5_K_M',
    'CHROMA': 'Q5_K_M',
    'WAN': 'Q5_K_M',
    'LTX': 'Q5_K_M',
}

def _floor_quant_key(qvar, size_dict):
    q = str(qvar).upper()
    for family, floor in MIN_QUANT_FLOORS.items():
        if family in q and floor in size_dict:
            return floor
    return _smallest_quant_key(size_dict)

def _reduce_quant_with_floor(qvar, current, size_dict):
    if not isinstance(size_dict, dict) or not size_dict or current not in size_dict:
        return current

    floor = _floor_quant_key(qvar, size_dict)
    if floor not in size_dict:
        floor = _smallest_quant_key(size_dict)

    cur_size = float(size_dict[current])
    floor_size = float(size_dict[floor])

    if cur_size <= floor_size:
        return current

    candidates = sorted((k, float(v)) for k, v in size_dict.items())
    step_down = [k for k, s in candidates if floor_size <= s < cur_size]
    if step_down:
        return step_down[-1]
    return floor


def _estimate_ram_peak_gb(g):
    # Selected quant sizes
    quant_sizes = {}
    for k, v in list(g.items()):
        if not (isinstance(k, str) and k.endswith('_QUANT') and isinstance(v, str)):
            continue
        dname = _size_dict_for_quant_var(k, g)
        if not dname:
            continue
        d = g[dname]
        if v in d:
            quant_sizes[k] = float(d[v])

    # Fixed model components from scalar *_GB constants
    fixed_components = []
    for k, v in list(g.items()):
        if not (isinstance(k, str) and k.endswith('_GB') and isinstance(v, (int, float))):
            continue
        if k.startswith('AUTO_'):
            continue
        if any(x in k for x in ['VRAM', 'RAM', 'BUDGET', 'FALLBACK', 'FRACTION', 'RESERVE', 'RUNTIME', 'SPILL', 'RESIDENT']):
            continue
        fixed_components.append(float(v))

    offload = bool(g.get('OFFLOAD_TEXT_ENCODER', g.get('OFFLOAD_TEXT_ENCODERS', False)))

    text_vals = []
    core_vals = []
    for qvar, size in quant_sizes.items():
        if _is_text_quant_var(qvar):
            text_vals.append(float(size))
        else:
            core_vals.append(float(size))

    if RAM_GUARD_ASSUME_SINGLE_MODEL_RUN:
        text_gb = max(text_vals) if text_vals else 0.0
        core_gb = max(core_vals) if core_vals else 0.0
    else:
        text_gb = sum(text_vals)
        core_gb = sum(core_vals)

    fixed_gb = sum(fixed_components)
    resident_proxy = core_gb + fixed_gb

    text_ram = text_gb * (AUTO_RAM_TEXT_OFFLOAD_FACTOR if offload else 0.20)
    spill_ram = resident_proxy * AUTO_RAM_SPILL_FACTOR
    resident_ram = resident_proxy * AUTO_RAM_GPU_RESIDENT_FACTOR

    ram_peak = AUTO_RAM_OS_RESERVE_GB + AUTO_RAM_RUNTIME_GB + text_ram + spill_ram + resident_ram
    return ram_peak, quant_sizes, text_gb, core_gb, fixed_gb


TOTAL_RAM_GB = detect_total_ram_gb()
RAM_BUDGET_GB = TOTAL_RAM_GB * AUTO_QUANT_RAM_FRACTION
RAM_PEAK_GB, _ram_quant_sizes, _ram_text_gb, _ram_core_gb, _ram_fixed_gb = _estimate_ram_peak_gb(globals())

print(f"\nRAM detected: ~{TOTAL_RAM_GB:.2f} GB | RAM budget ({int(AUTO_QUANT_RAM_FRACTION*100)}%): ~{RAM_BUDGET_GB:.2f} GB")
print(f"Estimated RAM peak: ~{RAM_PEAK_GB:.2f} GB (text~{_ram_text_gb:.2f}, core~{_ram_core_gb:.2f}, fixed~{_ram_fixed_gb:.2f})")

if AUTO_QUANT_BY_RAM and RAM_PEAK_GB > RAM_BUDGET_GB:
    print("RAM guard: estimated peak exceeds budget. Trying safer quant fallback with quality floors...")

    # 1) shrink text quants first
    for qvar in sorted(list(globals().keys())):
        if not (qvar.endswith('_QUANT') and _is_text_quant_var(qvar)):
            continue
        dname = _size_dict_for_quant_var(qvar, globals())
        if not dname:
            continue
        d = globals()[dname]
        if isinstance(d, dict) and d:
                current = globals().get(qvar)
                reduced = _reduce_quant_with_floor(qvar, current, d)
                if reduced is not None and current != reduced:
                    print(f"  RAM guard: {qvar} {current} -> {reduced}")
                    globals()[qvar] = reduced

    RAM_PEAK_GB, _, _, _, _ = _estimate_ram_peak_gb(globals())

    # 2) if still high, shrink remaining quants
    if RAM_PEAK_GB > RAM_BUDGET_GB:
        for qvar in sorted(list(globals().keys())):
            if not qvar.endswith('_QUANT'):
                continue
            if _is_text_quant_var(qvar):
                continue
            dname = _size_dict_for_quant_var(qvar, globals())
            if not dname:
                continue
            d = globals()[dname]
            if isinstance(d, dict) and d:
                current = globals().get(qvar)
                reduced = _reduce_quant_with_floor(qvar, current, d)
                if reduced is not None and current != reduced:
                    print(f"  RAM guard: {qvar} {current} -> {reduced}")
                    globals()[qvar] = reduced

    RAM_PEAK_GB, _, _, _, _ = _estimate_ram_peak_gb(globals())
    if RAM_PEAK_GB > RAM_BUDGET_GB:
        print(f"RAM guard result: still risky (~{RAM_PEAK_GB:.2f} GB > ~{RAM_BUDGET_GB:.2f} GB).")
    else:
        print(f"RAM guard result: OK (~{RAM_PEAK_GB:.2f} GB <= ~{RAM_BUDGET_GB:.2f} GB).")
else:
    print("RAM guard: OK (current selection fits estimated RAM budget).")


In [ ]:
# @title 3) Download models (FLUX.2-klein-9B base+distilled GGUF + Qwen3-8B GGUF + flux2 VAE + GGUF workflows)
import os
import json

COMFY = '/content/ComfyUI'

UNET_DIR = f'{COMFY}/models/unet'
DIFF_DIR = f'{COMFY}/models/diffusion_models'
TE_DIR   = f'{COMFY}/models/text_encoders'
CLIP_DIR = f'{COMFY}/models/clip'
VAE_DIR  = f'{COMFY}/models/vae'
WORKFLOW_DIR = f'{COMFY}/user/default/workflows'

for d in [UNET_DIR, DIFF_DIR, TE_DIR, CLIP_DIR, VAE_DIR, WORKFLOW_DIR]:
    os.makedirs(d, exist_ok=True)


def dl(url, outdir, fname):
    outpath = os.path.join(outdir, fname)
    if os.path.exists(outpath):
        print('Already exists:', outpath)
        return
    print('Downloading:', fname)

    hf_token = os.environ.get('HUGGINGFACE_TOKEN') or os.environ.get('HF_TOKEN')
    if hf_token and 'huggingface.co' in url:
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M --header="Authorization: Bearer {hf_token}" "{url}" -d "{outdir}" -o "{fname}"
    else:
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d "{outdir}" -o "{fname}"


def link_if_missing(src, dst):
    if not os.path.exists(dst):
        !ln -s "{src}" "{dst}"


# Base model GGUF
if DOWNLOAD_BASE_MODEL:
    base_fname = f"flux-2-klein-base-9b-{BASE_QUANT}.gguf"
    base_url = f"https://huggingface.co/unsloth/FLUX.2-klein-base-9B-GGUF/resolve/main/{base_fname}"
    dl(base_url, UNET_DIR, base_fname)
    link_if_missing(f"{UNET_DIR}/{base_fname}", f"{DIFF_DIR}/{base_fname}")
else:
    base_fname = None

# Distilled model GGUF
if DOWNLOAD_DISTILLED_MODEL:
    distilled_fname = f"flux-2-klein-9b-{DISTILLED_QUANT}.gguf"
    distilled_url = f"https://huggingface.co/unsloth/FLUX.2-klein-9B-GGUF/resolve/main/{distilled_fname}"
    dl(distilled_url, UNET_DIR, distilled_fname)
    link_if_missing(f"{UNET_DIR}/{distilled_fname}", f"{DIFF_DIR}/{distilled_fname}")
else:
    distilled_fname = None

# Quantized Qwen3-8B text encoder
q_fname = f"Qwen3-8B-{QWEN8_QUANT}.gguf"
q_url = f"https://huggingface.co/unsloth/Qwen3-8B-GGUF/resolve/main/{q_fname}"
dl(q_url, TE_DIR, q_fname)
link_if_missing(f"{TE_DIR}/{q_fname}", f"{CLIP_DIR}/{q_fname}")

# Official flux2 VAE
vae_fname = 'flux2-vae.safetensors'
vae_url = 'https://huggingface.co/Comfy-Org/flux2-dev/resolve/main/split_files/vae/flux2-vae.safetensors'
dl(vae_url, VAE_DIR, vae_fname)

# Download official templates then patch loader nodes to GGUF
wf_text_src = f"{WORKFLOW_DIR}/image_flux2_text_to_image_9b.json"
wf_edit_src = f"{WORKFLOW_DIR}/image_flux2_klein_image_edit_9b_base.json"
dl('https://raw.githubusercontent.com/Comfy-Org/workflow_templates/refs/heads/main/templates/image_flux2_text_to_image_9b.json', WORKFLOW_DIR, 'image_flux2_text_to_image_9b.json')
dl('https://raw.githubusercontent.com/Comfy-Org/workflow_templates/refs/heads/main/templates/image_flux2_klein_image_edit_9b_base.json', WORKFLOW_DIR, 'image_flux2_klein_image_edit_9b_base.json')


def patch_flux2_gguf_workflow(src_path, dst_path):
    with open(src_path, 'r', encoding='utf-8-sig') as f:
        wf = json.load(f)

    def patch_nodes(nodes, subgraph_name=''):
        for n in nodes:
            t = n.get('type')
            w = n.get('widgets_values')
            if not isinstance(w, list):
                continue

            if t == 'UNETLoader' and w and isinstance(w[0], str):
                name = w[0]
                if 'base-9b' in name:
                    w[0] = f"flux-2-klein-base-9b-{BASE_QUANT}.gguf"
                elif 'klein-9b' in name:
                    w[0] = f"flux-2-klein-9b-{DISTILLED_QUANT}.gguf"
                n['type'] = 'UnetLoaderGGUF'

            elif t == 'CLIPLoader' and w and isinstance(w[0], str):
                w[0] = f"Qwen3-8B-{QWEN8_QUANT}.gguf"
                n['type'] = 'CLIPLoaderGGUF'

            elif t == 'Flux2Scheduler' and w and isinstance(w[0], (int, float)):
                # keep distilled low-step behavior
                if 'Distilled' in (subgraph_name or ''):
                    w[0] = 4

    patch_nodes(wf.get('nodes', []), '')
    for s in wf.get('definitions', {}).get('subgraphs', []):
        patch_nodes(s.get('nodes', []), s.get('name', ''))

    with open(dst_path, 'w', encoding='utf-8') as f:
        json.dump(wf, f, ensure_ascii=False, indent=2)


def force_mode(src_path, dst_path, mode='base'):
    with open(src_path, 'r', encoding='utf-8-sig') as f:
        wf = json.load(f)

    for s in wf.get('definitions', {}).get('subgraphs', []):
        sname = s.get('name', '')
        for n in s.get('nodes', []):
            t = n.get('type')
            w = n.get('widgets_values')
            if not isinstance(w, list):
                continue
            if t == 'UnetLoaderGGUF' and w and isinstance(w[0], str):
                if mode == 'base':
                    w[0] = f"flux-2-klein-base-9b-{BASE_QUANT}.gguf"
                else:
                    w[0] = f"flux-2-klein-9b-{DISTILLED_QUANT}.gguf"
            elif t == 'Flux2Scheduler' and w and isinstance(w[0], (int, float)):
                w[0] = 20 if mode == 'base' else 4

    with open(dst_path, 'w', encoding='utf-8') as f:
        json.dump(wf, f, ensure_ascii=False, indent=2)


wf_gguf = f"{WORKFLOW_DIR}/image_flux2_text_to_image_9b_gguf.json"
patch_flux2_gguf_workflow(wf_text_src, wf_gguf)
patch_flux2_gguf_workflow(wf_edit_src, f"{WORKFLOW_DIR}/image_flux2_klein_image_edit_9b_base_gguf.json")

# Dedicated mode workflows
force_mode(wf_gguf, f"{WORKFLOW_DIR}/flux2_klein9b_base_t2i_gguf.json", mode='base')
force_mode(wf_gguf, f"{WORKFLOW_DIR}/flux2_klein9b_distilled_t2i_gguf.json", mode='distilled')

print('Done downloading models.')
if base_fname:
    print('Base model     :', base_fname)
if distilled_fname:
    print('Distilled model:', distilled_fname)
print('Text encoder   :', q_fname)
print('VAE            :', vae_fname)
print('Workflow       : flux2_klein9b_base_t2i_gguf.json (base, ~20 steps)')
print('Workflow       : flux2_klein9b_distilled_t2i_gguf.json (distilled, ~4 steps)')
print('Workflow       : image_flux2_klein_image_edit_9b_base_gguf.json')




In [ ]:
#@title 📥 Full Flux ve Özel 6 LoRA İndirici
#@markdown İndirmek istemediğiniz LoRA kutularını boş bırakın.
CIVITAI_TOKEN = "" #@param {type:"string"}
FLUX_FULL_LINK = "" #@param {type:"string"}
LORA_1 = "" #@param {type:"string"}
LORA_2 = "" #@param {type:"string"}
LORA_3 = "" #@param {type:"string"}
LORA_4 = "" #@param {type:"string"}
LORA_5 = "" #@param {type:"string"}
LORA_6 = "" #@param {type:"string"}

import os

checkpoint_dir = "/content/ComfyUI/models/checkpoints/"
lora_dir = "/content/ComfyUI/models/loras/"
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(lora_dir, exist_ok=True)

def indir(url, klasor, isim):
    if url.strip():
        print(f"İndiriliyor: {isim}...")
        os.system(f'aria2c --console-log-level=error -c -x 16 -s 16 -k 1M --header="Authorization: Bearer {CIVITAI_TOKEN}" "{url}" -d "{klasor}" -o "{isim}"')

if FLUX_FULL_LINK.strip():
    print("🚀 Tam Sürüm Flux Modeli İndiriliyor...")
    indir(FLUX_FULL_LINK, checkpoint_dir, "flux1-dev-full.safetensors")

loralar = [LORA_1, LORA_2, LORA_3, LORA_4, LORA_5, LORA_6]
for i, lora_url in enumerate(loralar):
    indir(lora_url, lora_dir, f"custom_lora_{i+1}.safetensors")

print("✅ Özel indirmeler tamamlandı!")

In [ ]:
# @title 4) Verify downloads: actual file sizes and VRAM estimate
import os


def size_gb(path):
    return os.path.getsize(path) / (1024**3)


def est_vram_from_file(path, overhead=1.10):
    return size_gb(path) * overhead


paths = []
if DOWNLOAD_BASE_MODEL:
    paths.append(('/content/ComfyUI/models/unet', f"flux-2-klein-base-9b-{BASE_QUANT}.gguf", 'base_model'))
if DOWNLOAD_DISTILLED_MODEL:
    paths.append(('/content/ComfyUI/models/unet', f"flux-2-klein-9b-{DISTILLED_QUANT}.gguf", 'distilled_model'))
paths.append(('/content/ComfyUI/models/text_encoders', f"Qwen3-8B-{QWEN8_QUANT}.gguf", 'text_encoder'))
paths.append(('/content/ComfyUI/models/vae', 'flux2-vae.safetensors', 'vae'))

print('Files:')
size_map = {}
for d, f, key in paths:
    p = os.path.join(d, f)
    if not os.path.exists(p):
        print('MISSING:', p)
        continue
    s = size_gb(p)
    v = est_vram_from_file(p)
    size_map[key] = v
    print(f"- {p}\n  size~{s:.2f} GB  -> VRAM(weights)~{v:.2f} GB")

shared = size_map.get('vae', 0.0)
if not OFFLOAD_TEXT_ENCODER:
    shared += size_map.get('text_encoder', 0.0)

if DOWNLOAD_BASE_MODEL and 'base_model' in size_map:
    print(f"\nEstimated GPU VRAM for BASE run: ~{shared + size_map['base_model']:.2f} GB")
if DOWNLOAD_DISTILLED_MODEL and 'distilled_model' in size_map:
    print(f"Estimated GPU VRAM for DISTILLED run: ~{shared + size_map['distilled_model']:.2f} GB")

print('Reminder: real peak VRAM is higher (resolution/batch/latents).')


In [ ]:
# @title 5) Launch ComfyUI (Cloudflare Tunnel) - stable mode
import subprocess, threading, time, socket, os, re, sys, urllib.request

%cd /content/ComfyUI

# Install cloudflared (if not installed yet)
if not os.path.exists("cloudflared-linux-amd64.deb"):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb


def wait_port(host="127.0.0.1", port=8188, timeout=240):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with socket.create_connection((host, port), timeout=1):
                return True
        except OSError:
            time.sleep(0.5)
    return False


def verify_tunnel_url(url, timeout=10):
    base = url.rstrip('/')
    candidates = [
        base + '/',
        base + '/api/system_stats',
        base + '/object_info',
    ]
    for c in candidates:
        try:
            req = urllib.request.Request(c, headers={'User-Agent': 'Mozilla/5.0'}, method='GET')
            with urllib.request.urlopen(req, timeout=timeout) as r:
                code = r.getcode()
                if 200 <= code < 500:
                    return True
        except Exception:
            pass
    return False


def wait_reachable_stable(url, proc, warmup_timeout=180, stable_successes=3, probe_interval=2.5):
    t0 = time.time()
    ok_streak = 0
    while time.time() - t0 < warmup_timeout:
        if proc.poll() is not None:
            return False
        if verify_tunnel_url(url, timeout=10):
            ok_streak += 1
            if ok_streak >= stable_successes:
                return True
        else:
            ok_streak = 0
        time.sleep(probe_interval)
    return False


def start_cloudflare_tunnel_once(port=8188, protocol='http2', read_timeout=150, warmup_timeout=180):
    print(f"\nStarting Cloudflare Quick Tunnel (protocol={protocol})...\n")
    sys.stdout.flush()

    cmd = [
        'cloudflared', 'tunnel',
        '--no-autoupdate',
        '--url', f'http://127.0.0.1:{port}',
        '--protocol', protocol,
        '--loglevel', 'info',
    ]

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    tunnel_patterns = [
        re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com', re.I),
        re.compile(r'https://[a-z0-9-]+\.cfargotunnel\.com', re.I),
    ]
    ignore_patterns = [
        re.compile(r'https://www\.cloudflare\.com/website-terms/?', re.I),
    ]

    t0 = time.time()
    url = None

    while time.time() - t0 < read_timeout:
        line = proc.stdout.readline()
        if not line:
            if proc.poll() is not None:
                break
            time.sleep(0.1)
            continue

        s = line.strip()

        if any(ip.search(s) for ip in ignore_patterns):
            continue

        for pat in tunnel_patterns:
            m = pat.search(s)
            if m:
                url = m.group(0)
                break
        if url:
            break

    if not url:
        print('Failed to parse tunnel URL.')
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=5)
            except Exception:
                proc.kill()
        return None, None

    print(f'Found tunnel URL: {url}')
    print('Waiting for Cloudflare propagation and stable readiness...')

    if wait_reachable_stable(url, proc, warmup_timeout=warmup_timeout, stable_successes=3, probe_interval=2.5):
        print('\n--------------------------------------------------')
        print('YOUR LINK:', url)
        print('--------------------------------------------------\n')
        return proc, url

    print('Tunnel URL did not become stably reachable in time.')
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()
    return None, url


def start_tunnel_with_retries(port=8188, max_attempts=6):
    protocols = ['http2', 'quic']
    for attempt in range(1, max_attempts + 1):
        proto = protocols[(attempt - 1) % len(protocols)]
        print(f"\n== Tunnel attempt {attempt}/{max_attempts} ==")
        proc, url = start_cloudflare_tunnel_once(
            port=port,
            protocol=proto,
            read_timeout=150,
            warmup_timeout=180,
        )
        if proc is not None and url:
            return proc, url
        time.sleep(min(8, 2 + attempt))
    return None, None


def stop_proc_safely(proc):
    if proc is None:
        return
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()


def tunnel_thread(port=8188):
    if not wait_port('127.0.0.1', port, timeout=240):
        print('Timed out waiting for ComfyUI port', port)
        return

    print('\nComfyUI port is open. Creating stable tunnel...\n')

    while True:
        proc, url = start_tunnel_with_retries(port=port, max_attempts=6)
        if proc is None:
            print('Failed to establish a reachable Cloudflare tunnel automatically.')
            print('Rerun this cell to retry with a fresh tunnel session.')
            return

        unhealthy_streak = 0
        while proc.poll() is None:
            time.sleep(15)
            if verify_tunnel_url(url, timeout=8):
                unhealthy_streak = 0
            else:
                unhealthy_streak += 1
                print(f'Cloudflare tunnel health check failed ({unhealthy_streak}/3).')
                if unhealthy_streak >= 3:
                    print('Tunnel became unhealthy. Recreating...')
                    stop_proc_safely(proc)
                    break

        if proc.poll() is not None:
            rc = proc.returncode
            print(f'Cloudflare tunnel exited (code={rc}). Recreating...')


threading.Thread(target=tunnel_thread, daemon=True, args=(8188,)).start()

print('Starting ComfyUI... link will appear above.')
!python main.py --dont-print-server --port 8188 --lowvram --preview-method auto
